# Pipeline Output Analysis

This notebook analyzes the combined mapper + verifier pipeline on gold UD sentences.

**Input:** `results/gold_ud_pipeline_meaningful.csv` (rows where `final_decision` is not `no_decision`)

**Source script:** `src/pipeline/run_gold_ud_pipeline.py` (first 50 train sentences)

Analysis only. No mapper, verifier, Stanza, or HTDB changes.

**Note:** `mapping_hypothesis` means an unverified mapper guess (UD label only). `confirmed` and `ambiguous` are verifier-backed.

**Questions:**
1. Are confirmed cases sensible?
2. Are ambiguous cases genuinely ambiguous?
3. Are mapping hypothesis cases useful or too weak?
4. Which mapper statuses and deprels drive mapping_hypothesis outputs?
5. What failure categories appear?
6. What should be refined later?

In [1]:
print("hello")

hello


## 1. Load the CSV

In [1]:
import csv
from collections import Counter, defaultdict
from pathlib import Path

CSV_PATH = Path("../results/gold_ud_pipeline_meaningful.csv")


def load_pipeline_csv(filepath):
    with open(filepath, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


rows = load_pipeline_csv(CSV_PATH)

print(f"Loaded {len(rows)} meaningful rows from {CSV_PATH}")
if rows:
    print(f"Columns: {', '.join(rows[0].keys())}")

Loaded 117 meaningful rows from ..\results\gold_ud_pipeline_meaningful.csv
Columns: sent_id, sentence_text, token_form, deprel, case_marker, mapper_candidates, mapper_confidence, mapper_status, verifier_candidates, verifier_decision, verifier_confidence, verifier_rule_id, final_candidates, final_decision, final_reason


## 2. Summary Counts

In [2]:
final_counts = Counter(row["final_decision"] for row in rows)
mapper_status_counts = Counter(row["mapper_status"] for row in rows)
deprel_counts = Counter(row["deprel"] for row in rows)
rule_counts = Counter(row["verifier_rule_id"] for row in rows if row["verifier_rule_id"])

print(f"Total meaningful rows: {len(rows)}")
print()

print("Count by final_decision:")
for decision, count in sorted(final_counts.items()):
    print(f"  {decision}: {count}")
print()

print("Count by mapper_status:")
for status, count in sorted(mapper_status_counts.items()):
    print(f"  {status}: {count}")
print()

print("Count by deprel:")
for deprel, count in sorted(deprel_counts.items(), key=lambda x: (-x[1], x[0])):
    print(f"  {deprel}: {count}")
print()

print("Count by verifier_rule_id:")
if rule_counts:
    for rule_id, count in sorted(rule_counts.items()):
        print(f"  {rule_id}: {count}")
else:
    print("  (none)")

Total meaningful rows: 117

Count by final_decision:
  ambiguous: 17
  confirmed: 19
  mapping_hypothesis: 81

Count by mapper_status:
  context_dependent: 62
  mapped: 55

Count by deprel:
  obl: 62
  nsubj: 40
  obj: 15

Count by verifier_rule_id:
  R1: 3
  R2: 15
  R3: 1
  R4: 11
  R5: 6


## 3. Helper: Display Examples

In [3]:
DISPLAY_COLS = [
    "sent_id",
    "token_form",
    "deprel",
    "case_marker",
    "final_decision",
    "final_candidates",
    "verifier_rule_id",
]


def show_examples(example_rows, title, max_examples=5, label_col=None):
    print(title)
    print("=" * 70)

    if not example_rows:
        print("(no rows)")
        print()
        return

    for i, row in enumerate(example_rows[:max_examples], start=1):
        label = ""
        if label_col and row.get(label_col):
            label = f" [{row[label_col]}]"
        print(f"Example {i}{label} ({row['sent_id']}, {row['token_form']})")
        print(f"  Sentence: {row['sentence_text']}")
        for col in DISPLAY_COLS:
            print(f"  {col}: {row[col]}")
        print(f"  mapper_status: {row['mapper_status']}")
        print(f"  mapper_candidates: {row['mapper_candidates']}")
        print()

## 4. Examples by final_decision

In [4]:
confirmed_rows = [r for r in rows if r["final_decision"] == "confirmed"]
ambiguous_rows = [r for r in rows if r["final_decision"] == "ambiguous"]
mapping_hypothesis_rows = [r for r in rows if r["final_decision"] == "mapping_hypothesis"]

show_examples(confirmed_rows, f"Confirmed rows ({len(confirmed_rows)} total)")
show_examples(ambiguous_rows, f"Ambiguous rows ({len(ambiguous_rows)} total)")
show_examples(mapping_hypothesis_rows, f"Mapping hypothesis rows ({len(mapping_hypothesis_rows)} total)")

Confirmed rows (19 total)
Example 1 (train-s2, शाहजेहन)
  Sentence: इसे नवाब शाहजेहन ने बनवाया था ।
  sent_id: train-s2
  token_form: शाहजेहन
  deprel: nsubj
  case_marker: ने
  final_decision: confirmed
  final_candidates: Kartā
  verifier_rule_id: R1
  mapper_status: mapped
  mapper_candidates: Kartā

Example 2 (train-s4, हॉल)
  Sentence: जिसमें चार मेहराबें हैं और मुख्य प्रार्थना हॉल में जाने के लिए 9 प्रवेश द्वार हैं ।
  sent_id: train-s4
  token_form: हॉल
  deprel: obl
  case_marker: में
  final_decision: confirmed
  final_candidates: Adhikaraṇa
  verifier_rule_id: R2
  mapper_status: context_dependent
  mapper_candidates: Adhikaraṇa|Apādāna|Karaṇa

Example 3 (train-s11, कोरिया)
  Sentence: इसे चार्ल्स कोरिया ने डिजाइन किया है ।
  sent_id: train-s11
  token_form: कोरिया
  deprel: nsubj
  case_marker: ने
  final_decision: confirmed
  final_candidates: Kartā
  verifier_rule_id: R1
  mapper_status: mapped
  mapper_candidates: Kartā

Example 4 (train-s12, क्षेत्र)
  Sentence: विशाल क्

## 5. Mapping Hypothesis Rows Grouped by deprel

In [5]:
mapper_by_deprel = defaultdict(list)
for row in mapping_hypothesis_rows:
    mapper_by_deprel[row["deprel"]].append(row)

print("mapping_hypothesis counts by deprel:")
for deprel in sorted(mapper_by_deprel, key=lambda d: (-len(mapper_by_deprel[d]), d)):
    group = mapper_by_deprel[deprel]
    status_counts = Counter(r["mapper_status"] for r in group)
    print(f"  {deprel}: {len(group)} rows | mapper_status: {dict(status_counts)}")
print()

for deprel in sorted(mapper_by_deprel, key=lambda d: (-len(mapper_by_deprel[d]), d)):
    show_examples(
        mapper_by_deprel[deprel],
        title=f"mapping_hypothesis / {deprel} ({len(mapper_by_deprel[deprel])} rows)",
        max_examples=3,
    )

mapping_hypothesis counts by deprel:
  nsubj: 37 rows | mapper_status: {'mapped': 37}
  obl: 35 rows | mapper_status: {'context_dependent': 35}
  obj: 9 rows | mapper_status: {'mapped': 9}

mapping_hypothesis / nsubj (37 rows)
Example 1 (train-s3, द्वार)
  Sentence: इसका प्रवेश द्वार दो मंजिला है ।
  sent_id: train-s3
  token_form: द्वार
  deprel: nsubj
  case_marker: 
  final_decision: mapping_hypothesis
  final_candidates: Kartā
  verifier_rule_id: 
  mapper_status: mapped
  mapper_candidates: Kartā

Example 2 (train-s4, द्वार)
  Sentence: जिसमें चार मेहराबें हैं और मुख्य प्रार्थना हॉल में जाने के लिए 9 प्रवेश द्वार हैं ।
  sent_id: train-s4
  token_form: द्वार
  deprel: nsubj
  case_marker: 
  final_decision: mapping_hypothesis
  final_candidates: Kartā
  verifier_rule_id: 
  mapper_status: mapped
  mapper_candidates: Kartā

Example 3 (train-s5, इमारत)
  Sentence: पूरी इमारत बेहद खूबसूरत है ।
  sent_id: train-s5
  token_form: इमारत
  deprel: nsubj
  case_marker: 
  final_decision: m

## 6. mapping_hypothesis: Mapper Status Breakdown

In [6]:
mapping_hypothesis_status = Counter(row["mapper_status"] for row in mapping_hypothesis_rows)
mapping_hypothesis_deprel_status = Counter(
    (row["deprel"], row["mapper_status"]) for row in mapping_hypothesis_rows
)

print("mapping_hypothesis by mapper_status:")
for status, count in sorted(mapping_hypothesis_status.items()):
    print(f"  {status}: {count}")
print()

print("mapping_hypothesis by (deprel, mapper_status):")
for key, count in sorted(mapping_hypothesis_deprel_status.items(), key=lambda x: (-x[1], x[0])):
    print(f"  {key[0]} + {key[1]}: {count}")

mapping_hypothesis by mapper_status:
  context_dependent: 35
  mapped: 46

mapping_hypothesis by (deprel, mapper_status):
  nsubj + mapped: 37
  obl + context_dependent: 35
  obj + mapped: 9


## 7. Manual Inspection Notes

Labels below are based on sentence context in the CSV. They are not gold Karaka annotations.

### 7.1 Strong confirmed cases

**Observation:** R1 and clear spatial R2/R3 hits look sensible in context.

| sent_id | Token | Rule | Note |
|---------|-------|------|------|
| train-s2 | शाहजेहन | R1 | Agent of बनवाया |
| train-s11 | कोरिया | R1 | Agent of डिजाइन किया |
| train-s50 | राजाओं | R1 | Agent of बनवाकर |
| train-s4 | हॉल | R2 | Location: prayer hall |
| train-s12 | क्षेत्र | R2 | Location: large area |
| train-s19 | संग्रहालय | R2 | Museum venue |
| train-s32 | चौक | R2 | Square location |
| train-s33 | गलियों | R2 | Lanes location |
| train-s36 | झील | R2 | Lake location |
| train-s15 | हिल्स | R3 | Surface location on hills |

**Hypothesis:** Verifier-confirmed rows are the most trustworthy part of the pipeline output.

### 7.2 Weak confirmed cases

**Observation:** Several R2 confirmations map में to Adhikaraṇa where the reading is extent, time, manner, or abstract.

| sent_id | Token | Issue |
|---------|-------|-------|
| train-s15 | एकड़ | Spatial extent (200 acres) |
| train-s26 | हैक्टेयर | Spatial extent (445 hectares) |
| train-s45 | वर्ष | Temporal frame |
| train-s48 | शताब्दी | Temporal frame |
| train-s49 | शताब्दी | Temporal frame |
| train-s34 | अंदाज | Manner (अंदाज में) |
| train-s50 | शान | Abstract locus (शान में) |

**Conclusion:** Confirmed does not always mean core physical location. R2 may over-confirm in v1.

### 7.3 Genuinely ambiguous cases

**Observation:** 6 ambiguous rows remain hard to resolve with light context.

| sent_id | Token | Rule | Reason |
|---------|-------|------|--------|
| train-s10 | रूप | R4 | Manner adverbial (मुख्य रूप से) |
| train-s17 | तरह | R4 | Manner adverbial (इस तरह से) |
| train-s17 | रूप | R4 | Manner adverbial (जीवंत रूप से) |
| train-s29 | रूप | R4 | Manner adverbial (मूल रूप से) |
| train-s6 | लोगों | R5 | आमंत्रित: recipient vs theme open |
| train-s47 | चमक | R5 | Abstract object; Karma vs Sampradāna unclear |

**Conclusion:** Pipeline correctly keeps these at ambiguous final_decision.

### 7.4 Potentially resolvable ambiguous cases

**Observation:** 11 ambiguous rows may support one Karaka with verb-frame context in v2.

**R4 examples:**
- train-s16 चित्रकला (सज्जित): material/instrument Karaṇa likely
- train-s20 हिस्सों, train-s23 जिलों (एकत्रित): Apādāna source likely
- train-s25 झील (से लगी), train-s35 झील/ओवरब्रिज (अलग): separation Apādāna likely
- train-s41 मुंबई (route origin): Apādāna likely

**R5 examples:**
- train-s19 पुस्तकालय, train-s23 नमूनों, train-s27 प्राणियों (perception/placement): Karma likely
- train-s48 वैभव (बयाँ करती): direct object Karma likely

**Hypothesis:** Verb-frame rules could refine some ambiguous rows without forcing all R5 cases.

### 7.5 Useful mapping hypothesis cases

**Observation:** 81 mapping_hypothesis rows fill gaps where the verifier returned no_decision. These are unverified mapper guesses, not verifier-backed Karaka decisions.

**Useful patterns:**

| Pattern | Count in batch | Example | Why useful |
|---------|----------------:|---------|------------|
| obj to Karma | 9 obj rows | इसे (train-s2), नमूने (train-s20) | Reasonable patient hypothesis when verifier is silent |
| obl locative without firing rule | some obl rows | यहाँ (train-s6) | Signals that a semantic role exists, but needs postposition/frame help |

**Hypothesis:** mapping_hypothesis obj rows provide a usable default Karma guess for downstream inspection.

### 7.6 Weak mapping hypothesis cases

**Observation:** Most mapping_hypothesis rows are nsubj (37 of 81) mapped to Kartā with low-medium confidence.

**Weak examples (copular / non-agent nsubj):**

| sent_id | Token | Issue |
|---------|-------|-------|
| train-s3 | द्वार | Subject of है, not an agent |
| train-s5 | इमारत | Stative copular subject |
| train-s8 | यह | Copular subject |
| train-s21 | संग्रहालय | Subject of बंद रहता है |
| train-s46 | समय | Abstract subject of लगता है |

**Weak context_dependent obl rows (35 of 81 mapping_hypothesis):**
- Triple candidate list (Adhikaraṇa|Apādāna|Karaṇa) with low confidence and no verifier refinement
- Examples: सोमवार obl+को (temporal), बजे obl+तक (endpoint), कैसे obl (interrogative)

**Conclusion:** mapping_hypothesis is informative but often weak. It should be read as a hypothesis label, not a verified Karaka assignment. Many nsubj rows look like future `corrected` candidates once mapper and verifier are compared explicitly.

### 7.7 Failure categories observed

| Category | Where it appears | Pipeline behavior |
|----------|-------------------|-------------------|
| R2 over-confirmation | Weak confirmed में hits | final_decision = confirmed |
| Copular nsubj as Kartā | mapping_hypothesis nsubj rows | mapping_hypothesis with Kartā |
| Manner से under R4 | रूप/तरह से rows | final_decision = ambiguous |
| Unresolved को under R5 | लोगों, चमक | final_decision = ambiguous |
| Vague obl fallback | obl without rule match | mapping_hypothesis with triple candidates |
| Mapper-verifier agreement on R1/R2/R3 | confirmed rows | mapper and verifier align on candidates |

**Note:** The pipeline does not yet emit `corrected` when mapper and verifier disagree (e.g. mapper Kartā on copular nsubj while verifier stays silent).

## 8. Summary

### What works

- **Confirmed R1** (3 rows): agent readings with ने look sensible.
- **Confirmed spatial R2/R3** (about 10 strong rows): classic location/surface cases look appropriate.
- **Ambiguous R4/R5** (17 rows): pipeline avoids forcing a single Karaka.
- **mapping_hypothesis obj** (9 rows): Karma default is a reasonable starting hypothesis (unverified).
- **Combined output schema**: mapper and verifier fields side by side support manual review.

### What is risky

- **Weak R2 confirmations** (7 rows): extent, time, manner, abstract में.
- **mapping_hypothesis nsubj to Kartā** (37 rows): many copular or non-agent subjects.
- **mapping_hypothesis context_dependent obl** (35 rows): triple candidates with low confidence.
- **No corrected decision type yet**: mapper can suggest Kartā while verifier stays silent.

### What to refine in v2 (hypotheses)

1. R2 subtype split for spatial vs temporal/extent/manner में.
2. R4 manner filter for रूप/तरह से (separate from Karaṇa|Apādāna).
3. Verb-frame disambiguation for resolvable R4/R5 rows.
4. `corrected` logic when mapper guesses Kartā on non-agent nsubj.
5. Narrower mapping_hypothesis presentation for context_dependent obl (or keep ambiguous).

### What not to change yet

- Core v1 verifier rules R1 to R5 (until dev-split evaluation is done).
- Conservative deprel-only mapper (postposition logic belongs in verifier).
- Priority rule: verifier confirmed/ambiguous overrides mapper.
- No Stanza, HTDB, or neural parser integration at this stage.

### Key counts (this CSV)

| Metric | Count |
|--------|------:|
| Meaningful rows | 117 |
| confirmed | 19 |
| ambiguous | 17 |
| mapping_hypothesis | 81 |
| mapping_hypothesis nsubj | 37 |
| mapping_hypothesis obl | 35 |
| mapping_hypothesis obj | 9 |